# Lost-feature grids: English + German + Human typicality (all cases, one notebook)

Extends `plots_lost_features_new_data.ipynb` (German, Human typicality) with a third,
**English** row, giving one **3-row x 4-column** figure per `(model, sae)` combination:

- **Row 1 -- English**: same a/b/c/d layout, sourced primarily from the new
  multi-case project (`experiment_1_set_consistency_new_data`, `data_case="english"`);
  where that's missing (English was rarely rerun there -- it's still mostly the original
  paper's data), it falls back to the three original single-dataset projects that
  `plots_lost_features.ipynb` reads (`experiment_1_set_consistency_paper` for panel a,
  `experiment_1_set_consistency_lost_fullseq_paper` for panels b)-d) main "(seq)" lines,
  `experiment_1_set_consistency_lost_paper` for the "(noun)" insets). This fallback is
  **English-only** -- German and Human typicality only ever existed in the new project.
- **Row 2 -- German**, **Row 3 -- Human typicality**: unchanged from
  `plots_lost_features_new_data.ipynb` (new project only, no old-project fallback --
  neither case existed before it).

Panel a) is lost rate vs. step `k` (with a per-layer inset) for English/German, and a
single lost-rate-vs-**layer** line (no k-dimension) for Human typicality. Panels b)-d) are
the "what happened to lost features" breakdowns; main "(seq)" lines from
`what_happened_to_lost_features_full_sequence.py` (or the old PROJECT_B for the English
fallback), "(noun)" insets from `what_happened_to_lost_features_noun.py` (or the old
PROJECT_C for the English fallback) -- omitted (not an error) wherever missing.

In [ ]:
# wandb's run-file listing (run.files()) breaks when a run's file count is an exact multiple of
# the page size (default 50): the paginator requests one page past the end and the API returns
# null -> "TypeError: 'NoneType' object is not subscriptable". The loaders below swallow
# exceptions per panel, so this silently dropped insets/panels from figures. List all files in a
# single page instead (per_page far above any run's file count), falling back to page sizes that
# don't divide typical counts. Returned as a list -- all callers here only iterate over it.
from wandb.apis.public import Run as _WandbRun

if not getattr(_WandbRun.files, "_single_page_patched", False):
    _orig_run_files = getattr(_WandbRun.files, "_orig", _WandbRun.files)

    def _run_files_single_page(self, names=None, per_page=None):
        last_err = None
        for pp in (10_000, 997, 37):
            try:
                return list(_orig_run_files(self, names=names or [], per_page=pp))
            except TypeError as e:
                last_err = e
        raise RuntimeError(f"run.files() failed for {'/'.join(self.path)}") from last_err

    _run_files_single_page._single_page_patched = True
    _run_files_single_page._orig = _orig_run_files
    _WandbRun.files = _run_files_single_page


In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from matplotlib.axes import Axes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

LABEL_FONTSIZE = 15
LEGEND_FONTSIZE = 11
TITLE_FONTSIZE = 14
TICK_FONTSIZE = 14
INSET_TICK_FONTSIZE = 11  # kept separate from TICK_FONTSIZE -- insets have little room to grow
ROW_HEIGHT = 4.0  # inches per row -- matches plots_lost_features_new_data.ipynb's 2-row figsize=(21.0, 8.0)

import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
ENTITY = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
PROJECT = "experiment_2_lost_latents"  # wandb.project in configs/experiment_2_lost_latents/base.yaml
SETUP_NAME = "A_fixed_base"
POOLING = "last_token"
MIX_MODE_SUFFIX = "lost_lasttok_signature_preact"
B_C_MODE: Any = 1

# The three original single-dataset (implicitly English) projects that
# plots_lost_features.ipynb reads -- used only as a fallback for the English row, only
# for (model, sae) combos that have no "english"-case run yet in PROJECT.
# Optional fallback for the English row from older single-dataset projects (as used for the paper,
# where some English runs predate the multi-dataset scripts). Leave as None when all runs are in PROJECT.
OLD_PROJECT_A = None
OLD_PROJECT_B = None
OLD_PROJECT_B_SWEEP = None
OLD_PROJECT_C = None
MIX_MODE_B = "lost_lasttok_signature_preact"

OUTPUT_DIR = Path("wandb_model_grids_all_data")
CACHE_DIR = Path(".wandb_plot_cache_all_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

JOB_TYPE_BY_SCRIPT = {
    "main_by_input_data.py": "lost_rate_by_input",
    "what_happened_to_lost_features_full_sequence.py": "lost_features_full_sequence",
    "what_happened_to_lost_features_noun.py": "lost_features_noun",
}

ROWS = [
    {"case_name": "english", "row_label": "English", "has_k": True},
    {"case_name": "german", "row_label": "German", "has_k": True},
    {"case_name": "human_typicality", "row_label": "Human\ntypicality", "has_k": False},
]

# gemma-3-270m/16k big is the only combo that ends up as a main-text figure in the paper;
# every other combo this notebook produces (including gemma-3-270m's other SAEs) is
# appendix material. Because 16k big's English row is already shown as its own figure in
# the main text, THIS notebook's copy of that row is replaced with a pointer back to it
# (see draw_reference_panel / plot_row) instead of repeating the data.
MAIN_TEXT_COMBO = ("gemma-3-270m", "16k big")
MAIN_TEXT_FIGURE_REF = "Fig. 4"

MODEL_ORDER = [
    "gemma-3-270m", "gemma-3-1b-pt", "gemma-3-4b-pt", "gemma-3-12b-pt", "gemma-3-27b-pt",
    "gpt2-small", "meta-llama/Llama-3.1-8B", "mistralai/Mistral-7B-v0.1",
]
GEMMA_MODELS = {"gemma-3-270m", "gemma-3-1b-pt", "gemma-3-4b-pt", "gemma-3-12b-pt", "gemma-3-27b-pt"}
GEMMA_INTERPRETABILITY_LAYERS: Dict[str, List[int]] = {
    "gemma-3-270m":  [5, 9, 12, 15],
    "gemma-3-1b-pt": [7, 13, 17, 22],
    "gemma-3-4b-pt": [9, 17, 22, 29],
    "gemma-3-12b-pt": [12, 24, 31, 41],
    "gemma-3-27b-pt": [16, 31, 40, 53],
}

## Identity / label helpers

Unchanged from `plots_lost_features_new_data.ipynb`.

In [ ]:
def cfg_get(cfg: Any, *keys: str, default=None):
    if not isinstance(cfg, dict):
        return default
    cur, ok = cfg, True
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            ok = False
            break
    return cur if ok else cfg.get(".".join(keys), default)


def safe_name(x: str) -> str:
    return x.replace("/", "__").replace(" ", "_")


def model_sort_key(x: str) -> int:
    return MODEL_ORDER.index(x) if x in MODEL_ORDER else 10**9


def get_run_name(run) -> str:
    return str(getattr(run, "name", "") or "")


def pretty_model_name(model_name: str) -> str:
    return {
        "gemma-3-270m": "Gemma 3 270M", "gemma-3-1b-pt": "Gemma 3 1B", "gemma-3-4b-pt": "Gemma 3 4B",
        "gemma-3-12b-pt": "Gemma 3 12B", "gemma-3-27b-pt": "Gemma 3 27B", "gpt2-small": "GPT-2 Small",
        "meta-llama/Llama-3.1-8B": "Llama 3.1 8B", "mistralai/Mistral-7B-v0.1": "Mistral 7B",
    }.get(model_name, model_name)


def title_sae_label(model_name: str, sae_label: str) -> str:
    '''Display-only simplification of sae_label for the figure suptitle. Gemma's
    regex-parsed labels ('16k big' etc.) are already short and returned as-is;
    the other releases carry their full SAE id (e.g. 'gpt2-small-resid-post-v5-32k
    blocks...') which is needed as-is for filenames/run lookups (sae_label itself
    is never touched) but is too long/ugly for the title. Substring checks (not
    \\\\b-bounded regex) because '_' counts as a word character, so e.g. \\\\b32x\\\\b
    never matches inside 'lxr_32x_l{}r_32x'.'''
    s = sae_label.lower()
    if model_name == "gpt2-small":
        m = re.search(r"(\d+)k", s)
        return f"{m.group(1)}K" if m else sae_label
    if model_name == "meta-llama/Llama-3.1-8B":
        if "_32x" in s:
            return "128K"  # 2**17 = 131,072 latents; called 128k by the release, like GPT-2 v5-128k
        if "_8x" in s:
            return "32K"
        return sae_label
    if model_name == "mistralai/Mistral-7B-v0.1":  # mistral-7b-res-wg: 65,536 latents
        return "65K"
    return sae_label


def get_model_name(run) -> Optional[str]:
    cfg = run.config or {}
    group = str(getattr(run, "group", "") or "")
    vals = [cfg_get(cfg, "sae", "model", "name"), cfg_get(cfg, "model", "name"), group]
    for v in vals:
        if v is None:
            continue
        s = str(v)
        if s in MODEL_ORDER:
            return s
        s_low = s.lower()
        if "gemma" in s_low and "270m" in s_low:
            return "gemma-3-270m"
        if "gemma" in s_low and "1b" in s_low:
            return "gemma-3-1b-pt"
        if "gemma" in s_low and "4b" in s_low:
            return "gemma-3-4b-pt"
        if "gemma" in s_low and "12b" in s_low:
            return "gemma-3-12b-pt"
        if "gemma" in s_low and "27b" in s_low:
            return "gemma-3-27b-pt"
        if "gpt2-small" in s_low:
            return "gpt2-small"
        if "llama-3.1-8b" in s_low:
            return "meta-llama/Llama-3.1-8B"
        if "mistral-7b" in s_low:
            return "mistralai/Mistral-7B-v0.1"
    return None


# Very early runs (before the SAE config was resolved into a full dict at log time) logged
# `run.config["sae"]` as a bare "<model>/<size>" override string instead of the structured SAE
# config -- so cfg_get(cfg, "sae", "release") finds nothing for them, and get_sae_short_label
# would otherwise fall back to the run's raw .name field, producing yet another distinct label
# for the exact same underlying SAE. Mapping each known legacy string to today's canonical
# release id keeps every run for a given SAE grouped together, regardless of when it was logged.
_LEGACY_SAE_LABEL_ALIASES: Dict[str, Dict[str, str]] = {
    "gpt2-small": {
        "gpt2-small/32k": "gpt2-small-resid-post-v5-32k",
        "gpt2-small/128k": "gpt2-small-resid-post-v5-128k",
    },
    "mistralai/Mistral-7B-v0.1": {
        "mistral-7b/65k": "mistral-7b-res-wg",
    },
    "meta-llama/Llama-3.1-8B": {
        # width labels map to the expansion-factor release ids: for Llama 3.1 8B's d_model=4096,
        # the "8x" release is the 32k-width SAE and "32x" the 128k-width one (2**17 latents;
        # configs/sae/llama-3_1-8b/128k.yaml, called 131k in older runs).
        "llama-3_1-8b/32k": "llama_scope_lxr_8x_l{}r_8x",
        "llama-3_1-8b/128k": "llama_scope_lxr_32x_l{}r_32x",
        "llama-3_1-8b/131k": "llama_scope_lxr_32x_l{}r_32x",
    },
}


def get_sae_short_label(run) -> Optional[str]:
    cfg = run.config or {}
    vals = [cfg_get(cfg, "sae", "release"), cfg_get(cfg, "sae", "name"), get_run_name(run)]
    for v in vals:
        if v is None:
            continue
        s = str(v).lower()
        for pat in [r"(\d+k)_(big|small|medium|canonical)", r"width[_ -](\d+k)(?:[_ -]l\d+)?[_ -](big|small|medium|canonical)\b"]:
            m = re.search(pat, s)
            if m:
                return f"{m.group(1)} {m.group(2)}"
    raw = cfg_get(cfg, "sae", "release")
    if raw is not None:
        label = str(raw).replace(".yaml", "").replace("_", " ").strip()
        return re.sub(r"\s+blocks\..*$", "", label).strip()

    sae_raw = cfg.get("sae")
    if isinstance(sae_raw, str):
        model_name = get_model_name(run)
        alias = _LEGACY_SAE_LABEL_ALIASES.get(model_name, {}).get(sae_raw.strip().lower())
        if alias:
            return alias
    return get_run_name(run)


def get_run_script(run) -> Optional[str]:
    jt = str(getattr(run, "job_type", "") or "")
    for script, job_type in JOB_TYPE_BY_SCRIPT.items():
        if jt == job_type:
            return script
    meta = run.metadata or {}
    program = str(meta.get("program", "") or meta.get("codePath", "") or "")
    for script in JOB_TYPE_BY_SCRIPT:
        if program.endswith(script):
            return script
    return None


def sae_sort_key(x: str):
    if x == "base":
        return (999, 999, x)
    m = re.match(r"(\d+)k(?:\s+(big|small|medium))?$", x)
    if m:
        size, var = int(m.group(1)), m.group(2)
        return (size, 0 if var == "big" else 1 if var == "medium" else 2, x)
    return (998, 998, x)


def sparse_integer_xticks(vals: List[int], n_ticks: int = 5) -> List[int]:
    vals = sorted(map(int, vals))
    if len(vals) <= n_ticks:
        return vals
    idxs = [round(i * (len(vals) - 1) / (n_ticks - 1)) for i in range(n_ticks)]
    ticks, seen = [], set()
    for idx in idxs:
        x = vals[idx]
        if x not in seen:
            ticks.append(x)
            seen.add(x)
    return ticks


def select_representative_layers(layers: List[int], n_select: int = 5) -> List[int]:
    layers = sorted(map(int, layers))
    if len(layers) <= n_select:
        return layers
    idxs = [round(i * (len(layers) - 1) / (n_select - 1)) for i in range(n_select)]
    out, seen = [], set()
    for idx in idxs:
        l = layers[idx]
        if l not in seen:
            out.append(l)
            seen.add(l)
    return out


def is_medium_sae(sae_label: str) -> bool:
    return bool(re.search(r"\bmedium\b", sae_label.lower()))


def select_gemma_layers(available_layers: List[int], sae_label: str, model_name: str) -> List[int]:
    available = set(available_layers)
    interp_layers = GEMMA_INTERPRETABILITY_LAYERS.get(model_name, [])
    interp = [l for l in interp_layers if l in available]
    if not is_medium_sae(sae_label) and 0 in available:
        interp = interp + [0]
    return interp

## New-project (multi-case) data loaders

Unchanged from `plots_lost_features_new_data.ipynb`.

In [ ]:
def parse_table_json(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return pd.DataFrame(obj["data"], columns=obj["columns"])


def download_file(file_obj, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    return Path(file_obj.download(root=str(out_dir), replace=True).name)


def find_case_table(run, *path_parts: str) -> Optional[Any]:
    prefix = "media/table/" + "/".join(path_parts)
    hits = [f for f in run.files() if f.name.startswith(prefix) and f.name.endswith(".table.json")]
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def load_panel_a_df(run, case_name: str) -> Optional[pd.DataFrame]:
    f = find_case_table(run, case_name, POOLING, "results_table")
    if f is None:
        return None
    path = download_file(f, CACHE_DIR / safe_name(get_model_name(run) or "unk") / safe_name(get_sae_short_label(run) or "unk") / "a" / case_name)
    df = parse_table_json(path)
    return pd.DataFrame({
        "layer": df["layer"].astype(int), "k": df["k"].astype(int),
        "mean": df["mean_lost_rate"].astype(float), "std": df["std_lost_rate"].astype(float),
        "setup_name": df["setup"].astype(str), "pooling": df["pooling"].astype(str),
    })


def load_metric_df(run, case_name: str, metric_name: str) -> Optional[pd.DataFrame]:
    f = find_case_table(run, case_name, MIX_MODE_SUFFIX, SETUP_NAME, f"{metric_name}_data")
    if f is None:
        return None
    path = download_file(f, CACHE_DIR / safe_name(get_model_name(run) or "unk") / safe_name(get_sae_short_label(run) or "unk") / "bc" / case_name)
    df = parse_table_json(path)
    df["layer"] = df["layer"].astype(int)
    df["k"] = df["k"].astype(int)
    df["mean"] = df["mean"].astype(float)
    df["std"] = df["std"].astype(float)
    return df


def df_to_plot_dicts(df: pd.DataFrame):
    df = df[(df["setup_name"] == SETUP_NAME) & (df["pooling"] == POOLING)].copy()
    mean_by_layer_by_k, std_by_layer_by_k = defaultdict(dict), defaultdict(dict)
    for _, row in df.iterrows():
        l, k = int(row["layer"]), int(row["k"])
        mean_by_layer_by_k[l][k] = float(row["mean"])
        std_by_layer_by_k[l][k] = float(row["std"])
    return dict(mean_by_layer_by_k), dict(std_by_layer_by_k), sorted(mean_by_layer_by_k.keys())


def panel_a_df_to_layer_dicts(df: pd.DataFrame, mode: Any):
    sub = df[(df["setup_name"] == SETUP_NAME) & (df["pooling"] == POOLING)].copy()
    target_k = 1 if mode == "aggregated" else int(mode)
    sub = sub[sub["k"] == target_k].copy()
    if sub.empty:
        raise ValueError(f"no rows for k={target_k}")
    layers = sorted(sub["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in sub.iterrows()}
    std_by_layer = {int(r["layer"]): float(r["std"]) for _, r in sub.iterrows()}
    return layers, mean_by_layer, std_by_layer


def metric_df_to_layer_dicts(df: pd.DataFrame, mode: Any):
    sub = df.copy()
    if mode == "aggregated":
        stats = sub.groupby("layer", as_index=False).agg(mean=("mean", "mean"), std=("mean", "std"))
        stats["std"] = stats["std"].fillna(0.0)
    else:
        k = int(mode)
        sub = sub[sub["k"] == k].copy()
        if sub.empty:
            raise ValueError(f"no rows for k={k}")
        stats = sub[["layer", "mean", "std"]].copy()
    layers = sorted(stats["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in stats.iterrows()}
    std_by_layer = {int(r["layer"]): float(r["std"]) for _, r in stats.iterrows()}
    return layers, mean_by_layer, std_by_layer

## Old-project English fallback

Only ever consulted for `case_name == "english"`, and only when the new project has
nothing for that `(model, sae)`. Three separate old projects, matching
`plots_lost_features.ipynb`:

- `OLD_PROJECT_A` -> panel a) (lost rate vs k)
- `OLD_PROJECT_B` (one sweep) -> panels b)-d) main "(seq)" lines
- `OLD_PROJECT_C` -> panels b)-d) "(noun)" insets

Each loader below normalizes its project's table schema down to the same
`layer/k/mean/std[/setup_name/pooling]` shape the new-project loaders already produce, so
the rest of the notebook (`metric_df_to_layer_dicts`, `panel_a_df_to_layer_dicts`, all the
`plot_*_on_ax` functions) stays completely agnostic to which project a row's data came from.

In [ ]:
def collect_old_project_a_runs() -> Dict[Tuple[str, str], Any]:
    '''Latest run per (model, sae) in OLD_PROJECT_A. This project predates the
    data_case split, so every run in it is implicitly "english".'''
    api = wandb.Api(timeout=60)
    runs = list(api.runs(f"{ENTITY}/{OLD_PROJECT_A}"))
    grouped: Dict[Tuple[str, str], List[Any]] = defaultdict(list)
    for run in runs:
        model_name = get_model_name(run)
        sae_label = get_sae_short_label(run)
        if model_name is None or sae_label is None:
            continue
        grouped[(model_name, sae_label)].append(run)
    return {k: sorted(v, key=lambda r: str(getattr(r, "created_at", "")))[-1] for k, v in grouped.items()}


def collect_old_project_runs(project: str, sweep_id: Optional[str] = None) -> Dict[Tuple[str, str], Any]:
    '''Latest run per (model, sae) in OLD_PROJECT_B or OLD_PROJECT_C. Mirrors
    plots_lost_features.ipynb's collect_runs/latest_run.'''
    api = wandb.Api(timeout=60)
    filters = {"sweep": sweep_id} if sweep_id is not None else {}
    runs = list(api.runs(f"{ENTITY}/{project}", filters=filters))
    grouped: Dict[Tuple[str, str], List[Any]] = defaultdict(list)
    for run in runs:
        model_name = get_model_name(run)
        sae_label = get_sae_short_label(run)
        if model_name is None or sae_label is None:
            continue
        grouped[(model_name, sae_label)].append(run)
    return {k: sorted(v, key=lambda r: str(getattr(r, "created_at", "")))[-1] for k, v in grouped.items()}


def find_old_table_file(run, contains: str):
    hits = [f for f in run.files() if contains in f.name and f.name.endswith(".table.json")]
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def find_old_metric_table_file(run, metric_name: str):
    target_prefix = f"{metric_name}_data_"
    hits = [f for f in run.files() if f.name.endswith(".table.json") and Path(f.name).name.startswith(target_prefix)]
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def load_old_panel_a_df(run) -> Optional[pd.DataFrame]:
    '''Same shape as load_panel_a_df, but OLD_PROJECT_A has no case-name path segment
    (media/table/{pooling}/results_table_... instead of
    media/table/{case_name}/{pooling}/results_table_...).'''
    f = find_old_table_file(run, "results_table")
    if f is None:
        return None
    path = download_file(f, CACHE_DIR / safe_name(get_model_name(run) or "unk") / safe_name(get_sae_short_label(run) or "unk") / "a_old_english")
    df = parse_table_json(path)
    return pd.DataFrame({
        "layer": df["layer"].astype(int), "k": df["k"].astype(int),
        "mean": df["mean_lost_rate"].astype(float), "std": df["std_lost_rate"].astype(float),
        "setup_name": df["setup"].astype(str), "pooling": df["pooling"].astype(str),
    })


def load_old_metric_df(run, metric_name: str, subdir: str) -> Optional[pd.DataFrame]:
    '''b)/c)/d) metric fallback for OLD_PROJECT_B ('(seq)' lines) / OLD_PROJECT_C
    ('(noun)' insets). Those projects don't scope a table to one setup/mix_mode via its
    file path the way the new project does, so that filtering has to happen here instead,
    down to the same layer/k/mean/std columns metric_df_to_layer_dicts expects.'''
    f = find_old_metric_table_file(run, metric_name)
    if f is None:
        return None
    path = download_file(f, CACHE_DIR / safe_name(get_model_name(run) or "unk") / safe_name(get_sae_short_label(run) or "unk") / subdir)
    df = parse_table_json(path)
    if "metric" in df.columns and "metric_name" not in df.columns:
        df = df.rename(columns={"metric": "metric_name"})
    df["layer"] = df["layer"].astype(int)
    df["k"] = df["k"].astype(int)
    df["mean"] = df["mean"].astype(float)
    df["std"] = df["std"].astype(float)
    df["setup_name"] = df["setup_name"].astype(str)
    df["mix_mode"] = df["mix_mode"].astype(str)
    sub = df[(df["setup_name"] == SETUP_NAME) & (df["mix_mode"] == MIX_MODE_B)]
    return sub[["layer", "k", "mean", "std"]].copy() if not sub.empty else None

Fetch the three old-project run indexes once, up front (reused for every English-row fallback lookup below).

In [ ]:
print(f"Fetching old-project run indexes (English fallback) ...")
OLD_A_RUNS = collect_old_project_a_runs() if OLD_PROJECT_A else {}
OLD_B_RUNS = collect_old_project_runs(OLD_PROJECT_B, OLD_PROJECT_B_SWEEP) if OLD_PROJECT_B else {}
OLD_C_RUNS = collect_old_project_runs(OLD_PROJECT_C) if OLD_PROJECT_C else {}
print(f"  OLD_PROJECT_A ({OLD_PROJECT_A}): {len(OLD_A_RUNS)} (model, sae) combos")
print(f"  OLD_PROJECT_B ({OLD_PROJECT_B}, sweep={OLD_PROJECT_B_SWEEP}): {len(OLD_B_RUNS)} (model, sae) combos")
print(f"  OLD_PROJECT_C ({OLD_PROJECT_C}): {len(OLD_C_RUNS)} (model, sae) combos")

## Plotting primitives

Unchanged from `plots_lost_features_new_data.ipynb`.

In [ ]:
def add_panel_letter(ax, letter: str) -> None:
    ax.text(-0.10, 0.99, letter, transform=ax.transAxes, ha="right", va="top", fontsize=18, fontweight="bold", clip_on=False)


def draw_blank(ax, subtitle: str = "nothing found") -> None:
    ax.text(0.5, 0.56, subtitle, ha="center", va="center", transform=ax.transAxes, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_alpha(0.3)


def draw_reference_panel(ax, letter: str, figure_ref: str = MAIN_TEXT_FIGURE_REF) -> None:
    """Used only for MAIN_TEXT_COMBO's English row: instead of repeating data already
    shown in the main-text figure, point back to it. One italicized line per panel,
    naming that panel's own letter (matches add_panel_letter's 'a)'/'b)'/... convention)."""
    ax.text(0.5, 0.5, f"see {figure_ref}{letter.rstrip(')')})", ha="center", va="center",
            transform=ax.transAxes, fontsize=12, style="italic")
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_alpha(0.3)


def plot_against_k_on_ax(ax, selected_layers, mean_by_layer_by_k, std_by_layer_by_k=None, ylabel="", k_values=None,
                          ylim=(-0.02, 1.02), show_legend=True, inlay_layers=None, mean_inlay_by_layer=None,
                          std_inlay_by_layer=None, inlay_loc="upper left", legend_loc="upper right"):
    layers = [int(x) for x in selected_layers]
    mean_int = {int(lk): {int(k): float(v) for k, v in d.items()} for lk, d in mean_by_layer_by_k.items()}
    std_int = {int(lk): {int(k): float(v) for k, v in d.items()} for lk, d in std_by_layer_by_k.items()} if std_by_layer_by_k else {}
    ks = sorted({k for layer in layers for k in mean_int.get(layer, {}).keys()}) if k_values is None else [int(k) for k in k_values]
    if not ks:
        draw_blank(ax, "no k data"); return
    xticks = sorted(set(int(round(x)) for x in np.linspace(ks[0], ks[-1], 5)))
    for layer in layers:
        x = ks
        y = [mean_int.get(layer, {}).get(k, float("nan")) for k in ks]
        line = ax.plot(x, y, linewidth=2.0, label=f"layer {layer}")
        color = line[0].get_color()
        if std_int:
            s = [std_int.get(layer, {}).get(k, 0.0) for k in ks]
            ax.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)], color=color, alpha=0.18, linewidth=0)
    ax.set_xticks(xticks)
    ax.set_xlabel("step $k$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.grid(True, linewidth=0.5, alpha=0.25)
    ax.set_ylim(*ylim); ax.set_yticks(np.linspace(0, 1, 11))
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc)
    if inlay_layers is not None and mean_inlay_by_layer is not None:
        x_in = [int(l) for l in inlay_layers]
        if x_in:
            axins = inset_axes(ax, width="24%", height="24%", loc=inlay_loc, bbox_to_anchor=(0.12, 0.02, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0.8)
            axins.patch.set_facecolor("white"); axins.patch.set_alpha(0.72)
            y_in = [float(mean_inlay_by_layer[l]) for l in x_in]
            s_in = [float(std_inlay_by_layer[l]) for l in x_in]
            line_in = axins.plot(x_in, y_in, marker="o", markersize=3, linewidth=1.6)
            color_in = line_in[0].get_color()
            axins.fill_between(x_in, [yy - ss for yy, ss in zip(y_in, s_in)], [yy + ss for yy, ss in zip(y_in, s_in)], color=color_in, alpha=0.18, linewidth=0)
            axins.set_xticks(sparse_integer_xticks(x_in, n_ticks=5))
            axins.set_xlim(min(x_in) - 0.5, max(x_in) + 0.5)
            axins.set_yticks([0, 0.5, 1.0]); axins.set_ylim(-0.02, 1.02)
            axins.tick_params(axis="both", labelsize=INSET_TICK_FONTSIZE, pad=2)
            axins.grid(True, alpha=0.25); axins.margins(x=0.06, y=0.08)


def plot_metric_against_layer_on_ax(ax, layers, mean_by_layer, std_by_layer, label="lost rate", ylabel="", color=None, legend_loc="upper right", show_legend=True,
                                     highlight_layers=None, highlight_colors=None):
    x = [int(l) for l in layers]
    y = [float(mean_by_layer[l]) for l in x]
    s = [float(std_by_layer.get(l, 0.0)) for l in x]
    line = ax.plot(x, y, marker="o", markersize=4, linewidth=2.0, label=label, color=color)
    c = line[0].get_color()
    ax.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)], color=c, alpha=0.18, linewidth=0)
    if highlight_layers:
        for i, l in enumerate(highlight_layers):
            if l in mean_by_layer:
                ax.plot(l, mean_by_layer[l], marker="o", markersize=9, linestyle="none",
                        color=highlight_colors[i % len(highlight_colors)], label=f"layer {l}", zorder=5)
    ax.set_xlabel("Layer $l$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(sparse_integer_xticks(x, n_ticks=5))
    ax.set_ylim(-0.02, 1.02); ax.set_yticks(np.linspace(0, 1, 11))
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc)


def plot_main_metric_on_ax(ax, layers, mean_main_by_layer, std_main_by_layer, main_label, ylabel, main_color=None, legend_loc="upper right", show_legend=True):
    x = [int(l) for l in layers]
    y = [float(mean_main_by_layer[l]) for l in x]
    s = [float(std_main_by_layer[l]) for l in x]
    line = ax.plot(x, y, marker="o", markersize=5, label=main_label, color=main_color)
    c = line[0].get_color()
    ax.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)], color=c, alpha=0.18, linewidth=0)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(sparse_integer_xticks(x, n_ticks=5))
    ax.set_yticks(np.linspace(0, 1, 11)); ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc, framealpha=0.9)


def plot_two_metrics_on_ax(ax, layers, mean1, std1, mean2, std2, label1, label2, ylabel, color1=None, color2=None, legend_loc="upper right", show_legend=True):
    x = [int(l) for l in layers]
    for mean_d, std_d, label, color in [(mean1, std1, label1, color1), (mean2, std2, label2, color2)]:
        y = [float(mean_d[l]) for l in x]
        s = [float(std_d[l]) for l in x]
        line = ax.plot(x, y, marker="o", markersize=5, linewidth=2.0, label=label, color=color)
        c = line[0].get_color()
        ax.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)], color=c, alpha=0.18, linewidth=0)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(sparse_integer_xticks(x, n_ticks=5))
    ax.set_xlim(min(x) - 0.5, max(x) + 0.5)
    ax.set_yticks(np.linspace(0, 1, 11)); ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc, framealpha=0.9)


def style_bc_axis(ax, last_row: bool, layers: List[int], xlabel=r"evaluation layer $\ell$"):
    ax.set_ylabel("fraction", fontsize=14)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(sparse_integer_xticks(layers, n_ticks=5))
    ax.set_xlim(min(layers) - 0.5, max(layers) + 0.5)
    ax.set_ylim(-0.02, 1.02); ax.set_yticks(np.linspace(0, 1, 11))
    if last_row:
        ax.set_xlabel(xlabel, fontsize=14)
    else:
        ax.set_xlabel(""); ax.tick_params(axis="x", labelbottom=False)


def _add_layer_inlay(ax, layers, means_list, stds_list, colors, loc="upper left", bbox_x=0.12):
    x = [int(l) for l in layers]
    if not x:
        return []
    axins = inset_axes(ax, width="24%", height="24%", loc=loc, bbox_to_anchor=(bbox_x, 0.02, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0.8)
    axins.patch.set_facecolor("white"); axins.patch.set_alpha(0.72)
    handles = []
    for mean_d, std_d, color in zip(means_list, stds_list, colors):
        y = [float(mean_d[l]) for l in x]
        s = [float(std_d[l]) for l in x]
        line = axins.plot(x, y, marker="o", markersize=3, linewidth=1.6, color=color)
        handles.append(line[0])
        axins.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)], color=color, alpha=0.18, linewidth=0)
    axins.set_xticks(sparse_integer_xticks(x, n_ticks=3))
    axins.set_xlim(min(x) - 0.5, max(x) + 0.5)
    axins.set_yticks([0, 0.5, 1.0]); axins.set_ylim(-0.02, 1.02)
    axins.tick_params(axis="both", labelsize=INSET_TICK_FONTSIZE, pad=2)
    axins.grid(True, alpha=0.25); axins.margins(x=0.06, y=0.08)
    return handles

## New-project run collection

In [ ]:
def collect_runs(project: str) -> Dict[Tuple[str, str, str], List[Any]]:
    api = wandb.Api(timeout=60)
    runs = list(api.runs(f"{ENTITY}/{project}"))
    grouped = defaultdict(list)
    for run in runs:
        model_name = get_model_name(run)
        sae_label = get_sae_short_label(run)
        script = get_run_script(run)
        if model_name is None or sae_label is None or script is None:
            continue
        grouped[(model_name, sae_label, script)].append(run)
    return {k: sorted(v, key=lambda r: str(getattr(r, "created_at", "")), reverse=True) for k, v in grouped.items()}


def first_non_empty_df(candidate_runs: List[Any], loader) -> Optional[pd.DataFrame]:
    for run in candidate_runs:
        try:
            df = loader(run)
        except Exception:
            df = None
        if df is not None and not df.empty:
            return df
    return None


def load_panel_a_df_any(candidate_runs: List[Any], case_name: str) -> Optional[pd.DataFrame]:
    return first_non_empty_df(candidate_runs, lambda r: load_panel_a_df(r, case_name))


def load_metric_df_any(candidate_runs: List[Any], case_name: str, metric_name: str) -> Optional[pd.DataFrame]:
    return first_non_empty_df(candidate_runs, lambda r: load_metric_df(r, case_name, metric_name))

## English-fallback-aware fetch layer

`fetch_panel_a_df`/`fetch_seq_metric_df`/`fetch_noun_metric_df` always try the new
project first; only for `case_name == "english"` do they fall back to the corresponding
old project when the new project has nothing for that `(model, sae)`. Everything
downstream (`load_three_metric_dfs`, `load_never_active_dfs`, `plot_row`) is unaware of
which source a dataframe came from.

In [ ]:
def fetch_panel_a_df(model_name: str, sae_label: str, case_name: str, runs_a_new: List[Any]) -> Optional[pd.DataFrame]:
    df = load_panel_a_df_any(runs_a_new, case_name)
    if df is not None and not df.empty:
        return df
    if case_name == "english":
        old_run = OLD_A_RUNS.get((model_name, sae_label))
        if old_run is not None:
            return load_old_panel_a_df(old_run)
    return None


def fetch_seq_metric_df(model_name: str, sae_label: str, case_name: str, runs_seq_new: List[Any], metric_name: str) -> Optional[pd.DataFrame]:
    df = load_metric_df_any(runs_seq_new, case_name, metric_name)
    if df is not None and not df.empty:
        return df
    if case_name == "english":
        old_run = OLD_B_RUNS.get((model_name, sae_label))
        if old_run is not None:
            return load_old_metric_df(old_run, metric_name, "bc_metric_old")
    return None


def fetch_noun_metric_df(model_name: str, sae_label: str, case_name: str, runs_noun_new: List[Any], metric_name: str) -> Optional[pd.DataFrame]:
    df = load_metric_df_any(runs_noun_new, case_name, metric_name)
    if df is not None and not df.empty:
        return df
    if case_name == "english":
        old_run = OLD_C_RUNS.get((model_name, sae_label))
        if old_run is not None:
            return load_old_metric_df(old_run, metric_name, "c_metric_old")
    return None


def load_three_metric_dfs(model_name: str, sae_label: str, case_name: str, runs_new: List[Any], fetch_fn, panel_name: str):
    main_df = fetch_fn(model_name, sae_label, case_name, runs_new, "prev_active_among_lost")
    in1_df = fetch_fn(model_name, sae_label, case_name, runs_new, "inactive_last_among_prev_active_lost")
    in2_df = fetch_fn(model_name, sae_label, case_name, runs_new, "negative_last_among_prev_active_lost")
    if main_df is None or in1_df is None or in2_df is None:
        raise ValueError(f"[{panel_name}] missing table(s) for case={case_name}")
    layers_main, mean_main, std_main = metric_df_to_layer_dicts(main_df, B_C_MODE)
    layers_in1, mean_in1, std_in1 = metric_df_to_layer_dicts(in1_df, B_C_MODE)
    layers_in2, mean_in2, std_in2 = metric_df_to_layer_dicts(in2_df, B_C_MODE)
    layers = sorted(set(layers_main) & set(layers_in1) & set(layers_in2))
    if not layers:
        raise ValueError(f"[{panel_name}] no overlapping layers")
    return (layers,
            {l: mean_main[l] for l in layers}, {l: std_main[l] for l in layers},
            {l: mean_in1[l] for l in layers}, {l: std_in1[l] for l in layers},
            {l: mean_in2[l] for l in layers}, {l: std_in2[l] for l in layers})


def load_never_active_dfs(model_name: str, sae_label: str, case_name: str, runs_new: List[Any], fetch_fn, panel_name: str):
    in1_df = fetch_fn(model_name, sae_label, case_name, runs_new, "always_inactive_among_never_prev_active_lost")
    in2_df = fetch_fn(model_name, sae_label, case_name, runs_new, "always_negative_among_never_prev_active_lost")
    if in1_df is None or in2_df is None:
        raise ValueError(f"[{panel_name}] missing table(s) for case={case_name}")
    layers_in1, mean_in1, std_in1 = metric_df_to_layer_dicts(in1_df, B_C_MODE)
    layers_in2, mean_in2, std_in2 = metric_df_to_layer_dicts(in2_df, B_C_MODE)
    layers = sorted(set(layers_in1) & set(layers_in2))
    if not layers:
        raise ValueError(f"[{panel_name}] no overlapping layers")
    return (layers, {l: mean_in1[l] for l in layers}, {l: std_in1[l] for l in layers},
            {l: mean_in2[l] for l in layers}, {l: std_in2[l] for l in layers})

## Row / grid drawing

In [ ]:
def plot_row(axes_row, runs_a, runs_seq, runs_noun, case_name: str, has_k: bool, model_name: str, sae_label: str,
             show_letters: bool, show_legend: bool, last_row: bool, row_label: str, use_reference_panels: bool = False):
    ax_a, ax_b, ax_c, ax_d = axes_row

    if show_letters:
        for ax, letter in zip(axes_row, ["a)", "b)", "c)", "d)"]):
            add_panel_letter(ax, letter)

    if use_reference_panels:
        # MAIN_TEXT_COMBO's English row: that data is already shown as its own main-text
        # figure, so this row (which would otherwise duplicate it inside an appendix
        # figure) is left blank with a pointer back to it instead -- no row_label either,
        # since there's nothing on this row to label.
        for ax, letter in zip(axes_row, ["a)", "b)", "c)", "d)"]):
            draw_reference_panel(ax, letter)
        return

    ax_a.text(-0.24, 0.5, row_label, transform=ax_a.transAxes, rotation=90, ha="center", va="center", fontsize=12, fontweight="bold", clip_on=False)

    # Panel a
    try:
        df_a = fetch_panel_a_df(model_name, sae_label, case_name, runs_a)
        if df_a is None or df_a.empty:
            raise ValueError("panel A table missing/empty")
        if has_k:
            mean_a, std_a, layers_a = df_to_plot_dicts(df_a)
            if model_name in GEMMA_MODELS:
                selected_layers = select_gemma_layers(list(mean_a.keys()), sae_label, model_name)
            else:
                selected_layers = select_representative_layers(layers_a, n_select=5)
            layers_inlay_a, mean_inlay_a, std_inlay_a = panel_a_df_to_layer_dicts(df_a, B_C_MODE)
            plot_against_k_on_ax(
                ax=ax_a, selected_layers=selected_layers,
                mean_by_layer_by_k={l: mean_a[l] for l in selected_layers},
                std_by_layer_by_k={l: std_a[l] for l in selected_layers},
                ylabel=r"lost rate $\mathcal{R}$", show_legend=show_legend,
                inlay_layers=layers_inlay_a, mean_inlay_by_layer=mean_inlay_a, std_inlay_by_layer=std_inlay_a,
            )
            if last_row:
                ax_a.set_xlabel(r"step $k$", fontsize=14)
        else:
            layers_a, mean_a, std_a = panel_a_df_to_layer_dicts(df_a, 1)
            if model_name in GEMMA_MODELS:
                highlight_layers = select_gemma_layers(layers_a, sae_label, model_name)
            else:
                highlight_layers = select_representative_layers(layers_a, n_select=5)
            highlight_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
            plot_metric_against_layer_on_ax(ax_a, layers_a, mean_a, std_a, label="lost rate",
                                             ylabel=r"lost rate $\mathcal{R}$", color="royalblue", show_legend=show_legend,
                                             highlight_layers=highlight_layers, highlight_colors=highlight_colors)
            if last_row:
                ax_a.set_xlabel(r"Layer $l$", fontsize=14)
    except Exception as e:
        print(f"[A] {case_name}/{sae_label} -> {e}")
        draw_blank(ax_a)

    # Panels b/c: main "(seq)" lines
    try:
        layers_b, mean_b, std_b, mean_b1, std_b1, mean_b2, std_b2 = load_three_metric_dfs(model_name, sae_label, case_name, runs_seq, fetch_seq_metric_df, "B")
        plot_main_metric_on_ax(ax_b, layers_b, mean_b, std_b, "detected upstream (seq)", "fraction", main_color="royalblue", show_legend=show_legend)
        style_bc_axis(ax_b, last_row, layers_b)
        plot_two_metrics_on_ax(ax_c, layers_b, mean_b1, std_b1, mean_b2, std_b2, "near-zero (seq)", "negative (seq)", "fraction", color1="red", color2="cyan", show_legend=show_legend)
        style_bc_axis(ax_c, last_row, layers_b)
    except Exception as e:
        print(f"[B/C] {case_name}/{sae_label} -> {e}")
        draw_blank(ax_b); draw_blank(ax_c)

    # Panels b/c: "(noun)" insets
    try:
        layers_n, mean_n, std_n, mean_n1, std_n1, mean_n2, std_n2 = load_three_metric_dfs(model_name, sae_label, case_name, runs_noun, fetch_noun_metric_df, "BC_noun")
        h_b = _add_layer_inlay(ax_b, layers_n, [mean_n], [std_n], ["darkorange"], bbox_x=0.06)
        h_c = _add_layer_inlay(ax_c, layers_n, [mean_n1, mean_n2], [std_n1, std_n2], ["magenta", "navy"])
        if show_legend:
            eh, el = ax_b.get_legend_handles_labels(); ax_b.legend(eh + h_b, el + ["detected upstream (noun)"], fontsize=LEGEND_FONTSIZE, loc="upper right", framealpha=0.9)
            eh, el = ax_c.get_legend_handles_labels(); ax_c.legend(eh + h_c, el + ["near-zero (noun)", "negative (noun)"], fontsize=LEGEND_FONTSIZE, loc="upper right", framealpha=0.9)
    except Exception as e:
        print(f"[B/C noun inset] {case_name}/{sae_label} -> {e}")

    # Panel d: main "(seq)"
    try:
        layers_d, mean_d1, std_d1, mean_d2, std_d2 = load_never_active_dfs(model_name, sae_label, case_name, runs_seq, fetch_seq_metric_df, "D")
        plot_two_metrics_on_ax(ax_d, layers_d, mean_d1, std_d1, mean_d2, std_d2, "always near-zero (seq)", "always negative (seq)", "fraction", color1="darkorange", color2="green", show_legend=show_legend)
        style_bc_axis(ax_d, last_row, layers_d)
    except Exception as e:
        print(f"[D] {case_name}/{sae_label} -> {e}")
        draw_blank(ax_d)

    # Panel d: "(noun)" inset
    try:
        layers_dn, mean_dn1, std_dn1, mean_dn2, std_dn2 = load_never_active_dfs(model_name, sae_label, case_name, runs_noun, fetch_noun_metric_df, "D_noun")
        h_d = _add_layer_inlay(ax_d, layers_dn, [mean_dn1, mean_dn2], [std_dn1, std_dn2], ["crimson", "purple"], bbox_x=0.06)
        if show_legend:
            eh, el = ax_d.get_legend_handles_labels()
            ax_d.legend(eh + h_d, el + ["always near-zero (noun)", "always negative (noun)"], fontsize=LEGEND_FONTSIZE, loc="upper right", framealpha=0.9)
    except Exception as e:
        print(f"[D noun inset] {case_name}/{sae_label} -> {e}")


def make_grid(model_name: str, sae_label: str, runs_a, runs_seq, runs_noun):
    runs_a_list = runs_a.get((model_name, sae_label, "main_by_input_data.py"), [])
    runs_seq_list = runs_seq.get((model_name, sae_label, "what_happened_to_lost_features_full_sequence.py"), [])
    runs_noun_list = runs_noun.get((model_name, sae_label, "what_happened_to_lost_features_noun.py"), [])
    if not runs_a_list and not runs_seq_list:
        return

    n_rows = len(ROWS)
    fig, axes = plt.subplots(n_rows, 4, figsize=(21.0, ROW_HEIGHT * n_rows), squeeze=False)
    fig.suptitle(f"{pretty_model_name(model_name)} — {title_sae_label(model_name, sae_label)}", fontsize=20, y=0.995)

    is_main_text_combo = (model_name, sae_label) == MAIN_TEXT_COMBO
    for i, row_cfg in enumerate(ROWS):
        plot_row(
            axes[i], runs_a_list, runs_seq_list, runs_noun_list,
            case_name=row_cfg["case_name"], has_k=row_cfg["has_k"],
            model_name=model_name, sae_label=sae_label,
            show_letters=(i == 0), show_legend=(i == 0), last_row=(i == n_rows - 1),
            row_label=row_cfg["row_label"],
            use_reference_panels=(is_main_text_combo and row_cfg["case_name"] == "english"),
        )

    fig.subplots_adjust(left=0.07, right=0.995, bottom=0.08, top=0.90, hspace=0.2, wspace=0.24)
    out_path = OUTPUT_DIR / f"{safe_name(model_name)}__{safe_name(sae_label)}__grid.png"
    fig.savefig(out_path, dpi=100, bbox_inches="tight")  # ~2100 px wide = ~300 dpi at full text width
    plt.close(fig)
    print(f"wrote {out_path}")

## Run it

In [ ]:
def main():
    runs_all = collect_runs(PROJECT)
    scripts_present = sorted({k[2] for k in runs_all.keys()})
    print("scripts present:", scripts_present)
    combos = sorted({(k[0], k[1]) for k in runs_all.keys()}, key=lambda mc: (model_sort_key(mc[0]), sae_sort_key(mc[1])))
    print("model/sae combos:", combos)
    for model_name, sae_label in combos:
        make_grid(model_name, sae_label, runs_all, runs_all, runs_all)

    return runs_all, combos


runs_all, combos = main()